# NOTEBOOK 05: Weerdata

Analyse en Voorbereiding van een externe weer dataset gevonden op: https://www.knmi.nl/nederland-nu/klimatologie/uurgegevens , en dan de uurgegevens van het station Heino tussen 3 maart 2024 en 11 september 2026

In [1]:
import pandas as pd

In [2]:
weer = pd.read_csv('../datasets/weerdata_heino.csv')


In [3]:
#Tijdsdata moet als STR staan om makkelijker te verwerken to Datetime

weer['YYYYMMDD'] = weer['YYYYMMDD'].astype(str).str.strip()
weer['HH'] = weer['HH'].astype(str).str.strip()


In [4]:
weer.info()

<class 'pandas.DataFrame'>
RangeIndex: 22032 entries, 0 to 22031
Data columns (total 25 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   STN       22032 non-null  int64
 1   YYYYMMDD  22032 non-null  str  
 2   HH        22032 non-null  str  
 3      DD     22032 non-null  str  
 4      FH     22032 non-null  str  
 5      FF     22032 non-null  str  
 6      FX     22032 non-null  str  
 7       T     22032 non-null  int64
 8    T10N     22032 non-null  str  
 9      TD     22032 non-null  str  
 10     SQ     22032 non-null  int64
 11      Q     22032 non-null  int64
 12     DR     22032 non-null  int64
 13     RH     22032 non-null  int64
 14      P     22032 non-null  str  
 15     VV     22032 non-null  str  
 16      N     22032 non-null  str  
 17      U     22032 non-null  str  
 18     WW     22032 non-null  str  
 19     IX     22032 non-null  str  
 20      M     22032 non-null  str  
 21      R     22032 non-null  str  
 22      S    

In [5]:
weer.isna().sum()

STN         0
YYYYMMDD    0
HH          0
   DD       0
   FH       0
   FF       0
   FX       0
    T       0
 T10N       0
   TD       0
   SQ       0
    Q       0
   DR       0
   RH       0
    P       0
   VV       0
    N       0
    U       0
   WW       0
   IX       0
    M       0
    R       0
    S       0
    O       0
    Y       0
dtype: int64

### Schoonmaken data

In [6]:
#Columns die niet gemeten woorden door het weerstation droppen

weer.columns = weer.columns.str.strip()
drop_cols = ['VV', 'STN', 'T10N', 'N', 'P', 'IX', 'M', 'WW', 'R', 'S', 'O', 'Y']
weer = weer.drop(columns=[c for c in drop_cols if c in weer.columns])

Het gaat hier om de kolommen: Minimumtemperatuur op 10cm (Deze wordt wel bijgehouden maar om de 6 uur), 
Horizontaal zicht, Bewolking, Relatieve luchtvochtigheid, Weercodeindicator, Mist, Regen, Sneeuw, Onweer en Ijsvorming.
Geen van deze wordt dus bijgehouden maar ze zijn ook allemaal waarschijnlijk niet van toepassing op de prestatie van het model.



In [7]:
weer.head()

,YYYYMMDD,HH,DD,FH,FF,FX,T,TD,SQ,Q,DR,RH,U
0,20240308,1,70,30,30,60,8,-6,0,0,0,0,90
1,20240308,2,70,30,20,50,2,-12,0,0,0,0,90
2,20240308,3,90,30,40,60,8,-10,0,0,0,0,87
3,20240308,4,90,30,30,50,5,-13,0,0,0,0,87
4,20240308,5,80,40,40,60,2,-16,0,0,0,0,87


In [8]:
#Hernoemen naar omschrijvende namen

weer = weer.rename(columns={
    'DD': 'Windrichting',
    'FH': 'GemWindsnelheid',
    'FF': 'WindSnelheid',
    'FX': 'Windstoot',
    'T': 'Temperatuur',
    'TD': 'Dauwtemperatuur',
    'SQ': 'DuurZonneschijn',
    'Q': 'GlobaleStraling',
    'DR': 'DuurNeerslag',
    'RH': 'SomNeerslag',
    'U': 'LuchtVochtigheid'
    
})

In [9]:
# Alle kolommen die numeriek horen te zijn

numeric_cols = [
    'Windrichting',
    'GemWindsnelheid',
    'WindSnelheid',
    'Windstoot',
    'Temperatuur',
    'Dauwtemperatuur',
    'DuurZonneschijn',
    'GlobaleStraling',
    'DuurNeerslag',
    'SomNeerslag',
    'LuchtVochtigheid'
]

for col in numeric_cols:
    if col in weer.columns:
        weer[col] = pd.to_numeric(weer[col], errors='coerce')


In [10]:
weer.dtypes

YYYYMMDD                str
HH                      str
Windrichting        float64
GemWindsnelheid     float64
WindSnelheid        float64
Windstoot           float64
Temperatuur           int64
Dauwtemperatuur     float64
DuurZonneschijn       int64
GlobaleStraling       int64
DuurNeerslag          int64
SomNeerslag           int64
LuchtVochtigheid    float64
dtype: object

In [11]:
# Omzetten van tienden eenheden naar hele eenheden

conversies = {
    'Temperatuur': 0.1,
    'Dauwtemperatuur': 0.1,
    'WindSnelheid': 0.1,
    'GemWindsnelheid': 0.1,
    'SomNeerslag': 0.1,
}

for col, factor in conversies.items():
    if col in weer.columns:
        weer[col] = weer[col] * factor
weer.head()

,YYYYMMDD,HH,Windrichting,GemWindsnelheid,WindSnelheid,Windstoot,Temperatuur,Dauwtemperatuur,DuurZonneschijn,GlobaleStraling,DuurNeerslag,SomNeerslag,LuchtVochtigheid
0,20240308,1,70.0,3.0,3.0,60.0,0.8,-0.6,0,0,0,0.0,90.0
1,20240308,2,70.0,3.0,2.0,50.0,0.2,-1.2,0,0,0,0.0,90.0
2,20240308,3,90.0,3.0,4.0,60.0,0.8,-1.0,0,0,0,0.0,87.0
3,20240308,4,90.0,3.0,3.0,50.0,0.5,-1.3,0,0,0,0.0,87.0
4,20240308,5,80.0,4.0,4.0,60.0,0.2,-1.6,0,0,0,0.0,87.0


### Omzetten van de Datum en Uur kolom naar een datetime

In [12]:
def clean_weather_timestamp_block(weer):
    import pandas as pd
# HH werkte met uren 1-24 ipv 0-23
    # HH leeg → 00
    mask_empty = (weer['HH'].isna()) | (weer['HH'].str.strip() == '')
    weer.loc[mask_empty, 'HH'] = '00'

    # HH = 24 → volgende dag + HH=00
    mask_24 = (weer['HH'] == '24')
    if mask_24.any():
        weer.loc[mask_24, 'YYYYMMDD'] = (
            pd.to_datetime(weer.loc[mask_24, 'YYYYMMDD'], format='%Y%m%d')
            + pd.Timedelta(days=1)
        ).dt.strftime('%Y%m%d')
        weer.loc[mask_24, 'HH'] = '00'

    # HH altijd 2 digits
    weer['HH'] = weer['HH'].str.zfill(2)

    # WINTERTIJD FIX (dubbele 02:00 → 03:00)
    dt = pd.to_datetime(weer['YYYYMMDD'] + weer['HH'], format='%Y%m%d%H')
    dupes = dt.duplicated(keep='first')
    dt.loc[dupes] = dt.loc[dupes] + pd.Timedelta(hours=1)

    # Timestamp aanmaken (timezone-aware)
    weer['timestamp'] = dt.dt.tz_localize(
        'Europe/Amsterdam',
        nonexistent='shift_forward',
        ambiguous='NaT'
    )

    # NaT verwijderen
    weer = weer.dropna(subset=['timestamp'])

    # Tijdkolommen verwijderen
    weer = weer.drop(columns=['YYYYMMDD', 'HH'])

    # Index zetten + duplicates verwijderen
    weer = weer.set_index('timestamp')
    weer = weer[~weer.index.duplicated(keep='first')]

    # KNMI-codes opschonen
    weer.loc[weer['Windrichting'] == 990, 'Windrichting'] = 0
    weer.loc[weer['SomNeerslag'] < 0, 'SomNeerslag'] = 0

    # Resample naar 5 minuten
    weer_5min = weer.resample('5min').ffill()

    # Timestamp terug als kolom
    weer_5min_reset = weer_5min.reset_index()

    return weer_5min_reset
weer_5min_reset = clean_weather_timestamp_block(weer)




### Opslaan als schone dataset

In [13]:

weer_5min_reset.to_csv('../datasets/weerdata_schoon.csv', index=False)

In [14]:
weer_5min_reset.info()

<class 'pandas.DataFrame'>
RangeIndex: 264361 entries, 0 to 264360
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype                           
---  ------            --------------   -----                           
 0   timestamp         264361 non-null  datetime64[us, Europe/Amsterdam]
 1   Windrichting      264049 non-null  float64                         
 2   GemWindsnelheid   263833 non-null  float64                         
 3   WindSnelheid      264049 non-null  float64                         
 4   Windstoot         264061 non-null  float64                         
 5   Temperatuur       264361 non-null  float64                         
 6   Dauwtemperatuur   262909 non-null  float64                         
 7   DuurZonneschijn   264361 non-null  int64                           
 8   GlobaleStraling   264361 non-null  int64                           
 9   DuurNeerslag      264361 non-null  int64                           
 10  SomNeerslag       2